<a href="https://colab.research.google.com/github/Janhvibabani/orbit-wars/blob/main/Reinforcement_Learning_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

orbit_wars_path = kagglehub.competition_download('orbit-wars')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0"

In [ ]:
from kaggle_environments import make

env = make("orbit_wars", debug=True)

In [ ]:
print(f"Environment: {env.name} v{env.version}")
print(f"Players: {env.specification.agents}")
print(f"Max steps: {env.configuration.episodeSteps}")

In [ ]:
%%writefile /content/default_cfg.yaml

seed: 42
run_name: orbit_wars_ppo
device: auto
save_dir: /content/artifacts
checkpoint_every: 50
log_every: 1
opponent: self
self_play_update_interval: 50
self_play_deterministic: false
alternate_player_sides: true

env:
  candidate_count: 8
  ship_bucket_count: 8

model:
  hidden_size: 128

ppo:
  rollout_steps: 64
  num_envs: 2
  total_updates: 10
  epochs: 4
  minibatch_size: 256
  gamma: 0.99
  clip_coef: 0.2
  ent_coef: 0.01
  vf_coef: 0.5
  lr: 0.0003
  max_grad_norm: 0.5

In [ ]:
# config.py
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import yaml


@dataclass(slots=True)
class EnvConfig:
    board_size: float = 100.0
    episode_steps: int = 500
    candidate_count: int = 8
    ship_bucket_count: int = 8
    max_planets: int = 48
    max_ships: float = 400.0
    max_production: float = 5.0


@dataclass(slots=True)
class ModelConfig:
    hidden_size: int = 128


@dataclass(slots=True)
class PPOConfig:
    rollout_steps: int = 32
    num_envs: int = 4
    total_updates: int = 200
    epochs: int = 4
    minibatch_size: int = 512
    gamma: float = 0.99
    clip_coef: float = 0.2
    ent_coef: float = 0.01
    vf_coef: float = 0.5
    lr: float = 3e-4
    max_grad_norm: float = 0.5


@dataclass(slots=True)
class TrainConfig:
    seed: int = 42
    run_name: str = "orbit_wars_template_ppo"
    device: str = "auto"
    save_dir: str = "artifacts/rl_template"
    checkpoint_every: int = 10
    log_every: int = 1
    opponent: str = "random"
    self_play_update_interval: int = 10
    self_play_deterministic: bool = False
    alternate_player_sides: bool = True
    env: EnvConfig = field(default_factory=EnvConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    ppo: PPOConfig = field(default_factory=PPOConfig)


def default_train_config_path() -> Path:
    return Path(__file__).resolve().parent / "configs" / "default.yaml"


def load_train_config(path: str | Path) -> TrainConfig:
    config_path = Path(path)
    data = yaml.safe_load(config_path.read_text(encoding="utf-8")) or {}
    if not isinstance(data, dict):
        raise ValueError(f"YAML config must be a mapping: {config_path}")
    return train_config_from_dict(data)


def train_config_from_dict(data: dict[str, Any]) -> TrainConfig:
    cfg = TrainConfig()
    _update_dataclass(cfg, data, skip={"env", "model", "ppo"})
    _update_dataclass(cfg.env, data.get("env", {}))
    _update_dataclass(cfg.model, data.get("model", {}))
    _update_dataclass(cfg.ppo, data.get("ppo", {}))
    return cfg


def _update_dataclass(instance: Any, values: dict[str, Any], skip: set[str] | None = None) -> None:
    if not isinstance(values, dict):
        return
    skip = skip or set()
    for key, value in values.items():
        if key in skip or not hasattr(instance, key):
            continue
        default = getattr(instance, key)
        setattr(instance, key, _coerce_value(value, default))


def _coerce_value(value: Any, default: Any) -> Any:
    if isinstance(default, bool):
        if isinstance(value, str):
            lowered = value.strip().lower()
            if lowered in {"1", "true", "yes", "on"}:
                return True
            if lowered in {"0", "false", "no", "off"}:
                return False
        return bool(value)
    if isinstance(default, int) and not isinstance(default, bool):
        return int(value)
    if isinstance(default, float):
        return float(value)
    return value

In [ ]:
# game_types.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Iterable

@dataclass(slots=True)
class PlanetState:
    id: int
    owner: int
    x: float
    y: float
    radius: float
    ships: int
    production: int


@dataclass(slots=True)
class FleetState:
    id: int
    owner: int
    x: float
    y: float
    angle: float
    from_planet_id: int
    ships: int


@dataclass(slots=True)
class GameState:
    step: int
    player: int
    planets: list[PlanetState]
    fleets: list[FleetState]


def _safe_get(obj: Any, key: str, default: Any) -> Any:
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def _as_rows(data: Any) -> Iterable:
    return data if data is not None else []


def parse_planets(rows: Any) -> list[PlanetState]:
    return [
        PlanetState(
            id=int(r[0]),
            owner=int(r[1]),
            x=float(r[2]),
            y=float(r[3]),
            radius=float(r[4]),
            ships=int(r[5]),
            production=int(r[6]),
        )
        for r in _as_rows(rows)
    ]


def parse_fleets(rows: Any) -> list[FleetState]:
    return [
        FleetState(
            id=int(r[0]),
            owner=int(r[1]),
            x=float(r[2]),
            y=float(r[3]),
            angle=float(r[4]),
            from_planet_id=int(r[5]),
            ships=int(r[6]),
        )
        for r in _as_rows(rows)
    ]


def parse_observation(observation: Any) -> GameState:
    planets_raw = _safe_get(observation, "planets", [])
    fleets_raw = _safe_get(observation, "fleets", [])

    return GameState(
        step=int(_safe_get(observation, "step", 0)),
        player=int(_safe_get(observation, "player", 0)),
        planets=parse_planets(planets_raw),
        fleets=parse_fleets(fleets_raw),
    )

In [ ]:
# features.py
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any

import numpy as np

# from .config import EnvConfig
# from .game_types import GameState, PlanetState, parse_observation

BOARD_CENTER = (50.0, 50.0)
ROTATION_RADIUS_LIMIT = 50.0
SUN_RADIUS = 10.0
PLANET_LAUNCH_RADIUS_OFFSET = 0.1


@dataclass(slots=True)
class DecisionContext:
    env_index: int
    source_id: int
    candidate_ids: list[int]
    candidate_mask: np.ndarray
    ship_counts: list[int]
    target_angles: list[float]


@dataclass(slots=True)
class TurnBatch:
    self_features: np.ndarray
    candidate_features: np.ndarray
    global_features: np.ndarray
    candidate_mask: np.ndarray
    contexts: list[DecisionContext]
    state: GameState


def self_feature_dim() -> int:
    return 11


def candidate_feature_dim() -> int:
    return 14


def global_feature_dim() -> int:
    return 8


def encode_turn(
    observation: Any,
    env_cfg: EnvConfig,
    *,
    env_index: int = 0,
) -> TurnBatch:
    state = observation if isinstance(observation, GameState) else parse_observation(observation)
    my_planets = sorted((planet for planet in state.planets if planet.owner == state.player), key=lambda planet: planet.id)
    if not my_planets:
        return TurnBatch(
            self_features=np.zeros((0, self_feature_dim()), dtype=np.float32),
            candidate_features=np.zeros((0, env_cfg.candidate_count, candidate_feature_dim()), dtype=np.float32),
            global_features=np.zeros((0, global_feature_dim()), dtype=np.float32),
            candidate_mask=np.zeros((0, env_cfg.candidate_count), dtype=bool),
            contexts=[],
            state=state,
        )

    global_feat = build_global_features(state, env_cfg)
    self_rows: list[np.ndarray] = []
    candidate_rows: list[np.ndarray] = []
    candidate_masks: list[np.ndarray] = []
    contexts: list[DecisionContext] = []

    for src in my_planets:
        candidates = build_candidates(src, state, env_cfg)
        cand_feat, cand_mask, ship_counts, candidate_ids, target_angles = build_candidate_features(
            src,
            candidates,
            state,
            env_cfg,
        )
        self_rows.append(build_self_features(src, state, env_cfg))
        candidate_rows.append(cand_feat)
        candidate_masks.append(cand_mask)
        contexts.append(
            DecisionContext(
                env_index=env_index,
                source_id=src.id,
                candidate_ids=candidate_ids,
                candidate_mask=cand_mask,
                ship_counts=ship_counts,
                target_angles=target_angles,
            )
        )

    return TurnBatch(
        self_features=np.asarray(self_rows, dtype=np.float32),
        candidate_features=np.asarray(candidate_rows, dtype=np.float32),
        global_features=np.repeat(global_feat[None, :], len(self_rows), axis=0),
        candidate_mask=np.asarray(candidate_masks, dtype=bool),
        contexts=contexts,
        state=state,
    )

def build_candidates(src: PlanetState, state: GameState, env_cfg: EnvConfig) -> list[PlanetState]:
    others = [planet for planet in state.planets if planet.id != src.id]
    enemy_quota = env_cfg.candidate_count // 3
    neutral_quota = env_cfg.candidate_count // 3
    friendly_quota = env_cfg.candidate_count - enemy_quota - neutral_quota

    enemies = sorted(
        (planet for planet in others if planet.owner not in {-1, state.player}),
        key=lambda planet: (distance(src, planet), planet.id),
    )[:enemy_quota]
    neutrals = sorted(
        (planet for planet in others if planet.owner == -1),
        key=lambda planet: (distance(src, planet), planet.id),
    )[:neutral_quota]
    friendlies = sorted(
        (planet for planet in others if planet.owner == state.player),
        key=lambda planet: (distance(src, planet), planet.id),
    )[:friendly_quota]

    selected_ids = {planet.id for planet in enemies + neutrals + friendlies}
    candidates = enemies + neutrals + friendlies
    if len(candidates) >= env_cfg.candidate_count:
        return candidates[: env_cfg.candidate_count]

    fallback = sorted(
        (planet for planet in others if planet.id not in selected_ids),
        key=lambda planet: (distance(src, planet), planet.id),
    )
    candidates.extend(fallback[: env_cfg.candidate_count - len(candidates)])
    return candidates


def build_self_features(src: PlanetState, state: GameState, env_cfg: EnvConfig) -> np.ndarray:
    my_planets = [planet for planet in state.planets if planet.owner == state.player]
    enemy_planets = [planet for planet in state.planets if planet.owner not in {-1, state.player}]
    return np.asarray(
        [
            1.0,
            src.x / env_cfg.board_size,
            src.y / env_cfg.board_size,
            src.radius / 5.0,
            min(src.ships, env_cfg.max_ships) / env_cfg.max_ships,
            src.production / env_cfg.max_production,
            1.0 if is_rotating_planet(src) else 0.0,
            len(my_planets) / env_cfg.max_planets,
            len(enemy_planets) / env_cfg.max_planets,
            total_ships(my_planets) / (env_cfg.max_planets * env_cfg.max_ships),
            total_ships(enemy_planets) / (env_cfg.max_planets * env_cfg.max_ships),
        ],
        dtype=np.float32,
    )


def build_candidate_features(
    src: PlanetState,
    candidates: list[PlanetState],
    state: GameState,
    env_cfg: EnvConfig,
) -> tuple[np.ndarray, np.ndarray, list[int], list[int], list[float]]:
    features = np.zeros((env_cfg.candidate_count, candidate_feature_dim()), dtype=np.float32)
    candidate_mask = np.zeros((env_cfg.candidate_count,), dtype=bool)
    ship_counts = [0] * env_cfg.candidate_count
    candidate_ids = [-1] * env_cfg.candidate_count
    target_angles = [0.0] * env_cfg.candidate_count
    candidate_mask[0] = True

    for idx, tgt in enumerate(candidates, start=1):
        if idx >= env_cfg.candidate_count:
            break
        dx = tgt.x - src.x
        dy = tgt.y - src.y
        angle = math.atan2(dy, dx)
        crosses_sun = shot_crosses_sun(src, angle, tgt)
        ships_needed = fixed_ship_count(src, tgt)
        features[idx] = np.asarray(
            [
                1.0,
                1.0 if tgt.owner == -1 else 0.0,
                1.0 if tgt.owner == state.player else 0.0,
                1.0 if tgt.owner not in {-1, state.player} else 0.0,
                tgt.x / env_cfg.board_size,
                tgt.y / env_cfg.board_size,
                dx / env_cfg.board_size,
                dy / env_cfg.board_size,
                distance(src, tgt) / env_cfg.board_size,
                min(tgt.ships, env_cfg.max_ships) / env_cfg.max_ships,
                tgt.production / env_cfg.max_production,
                1.0 if is_rotating_planet(tgt) else 0.0,
                1.0 if crosses_sun else 0.0,
                min(src.ships, env_cfg.max_ships) / env_cfg.max_ships,
            ],
            dtype=np.float32,
        )
        ship_counts[idx] = ships_needed
        candidate_mask[idx] = ships_needed > 0 and not crosses_sun and src.ships >= ships_needed
        candidate_ids[idx] = tgt.id
        target_angles[idx] = angle

    return features, candidate_mask, ship_counts, candidate_ids, target_angles


def build_global_features(state: GameState, env_cfg: EnvConfig) -> np.ndarray:
    my_planets = [planet for planet in state.planets if planet.owner == state.player]
    enemy_planets = [planet for planet in state.planets if planet.owner not in {-1, state.player}]
    neutral_planets = [planet for planet in state.planets if planet.owner == -1]
    my_fleets = [fleet for fleet in state.fleets if fleet.owner == state.player]
    enemy_fleets = [fleet for fleet in state.fleets if fleet.owner != state.player]
    return np.asarray(
        [
            state.step / env_cfg.episode_steps,
            len(my_planets) / env_cfg.max_planets,
            len(enemy_planets) / env_cfg.max_planets,
            len(neutral_planets) / env_cfg.max_planets,
            total_ships(my_planets) / (env_cfg.max_planets * env_cfg.max_ships),
            total_ships(enemy_planets) / (env_cfg.max_planets * env_cfg.max_ships),
            sum(fleet.ships for fleet in my_fleets) / (env_cfg.max_planets * env_cfg.max_ships),
            sum(fleet.ships for fleet in enemy_fleets) / (env_cfg.max_planets * env_cfg.max_ships),
        ],
        dtype=np.float32,
    )


def fixed_ship_count(src: PlanetState, tgt: PlanetState) -> int:
    return max(tgt.ships + 1, 20)


def distance(a: PlanetState, b: PlanetState) -> float:
    return math.hypot(a.x - b.x, a.y - b.y)


def total_ships(planets: list[PlanetState]) -> float:
    return float(sum(planet.ships for planet in planets))


def is_rotating_planet(planet: PlanetState) -> bool:
    dx = planet.x - BOARD_CENTER[0]
    dy = planet.y - BOARD_CENTER[1]
    orbital_radius = math.hypot(dx, dy)
    return orbital_radius + planet.radius < ROTATION_RADIUS_LIMIT


def shot_crosses_sun(src: PlanetState, angle: float, tgt: PlanetState) -> bool:
    start_x = src.x + math.cos(angle) * (src.radius + PLANET_LAUNCH_RADIUS_OFFSET)
    start_y = src.y + math.sin(angle) * (src.radius + PLANET_LAUNCH_RADIUS_OFFSET)
    return point_to_segment_distance(BOARD_CENTER, (start_x, start_y), (tgt.x, tgt.y)) < SUN_RADIUS


def point_to_segment_distance(point: tuple[float, float], start: tuple[float, float], end: tuple[float, float]) -> float:
    segment_len_sq = (start[0] - end[0]) ** 2 + (start[1] - end[1]) ** 2
    if segment_len_sq == 0.0:
        return math.hypot(point[0] - start[0], point[1] - start[1])
    projection = (
        ((point[0] - start[0]) * (end[0] - start[0]) + (point[1] - start[1]) * (end[1] - start[1]))
        / segment_len_sq
    )
    projection = max(0.0, min(1.0, projection))
    closest_x = start[0] + projection * (end[0] - start[0])
    closest_y = start[1] + projection * (end[1] - start[1])
    return math.hypot(point[0] - closest_x, point[1] - closest_y)

In [ ]:
# policy.py
from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn as nn


@dataclass(slots=True)
class PolicyOutput:
    target_logits: torch.Tensor
    value: torch.Tensor


class PlanetPolicy(nn.Module):
    def __init__(
        self,
        self_dim: int,
        candidate_dim: int,
        global_dim: int,
        candidate_count: int,
        hidden_size: int = 128,
    ) -> None:
        super().__init__()
        self.candidate_count = candidate_count
        self.self_encoder = nn.Sequential(
            nn.Linear(self_dim, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
        )
        self.global_encoder = nn.Sequential(
            nn.Linear(global_dim, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
        )
        self.candidate_encoder = nn.Sequential(
            nn.Linear(candidate_dim, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
        )
        self.target_head = nn.Sequential(
            nn.Linear(hidden_size * 3, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),
        )
        self.value_head = nn.Sequential(
            nn.Linear(hidden_size * 3, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),
        )

    def forward(
        self,
        self_features: torch.Tensor,
        candidate_features: torch.Tensor,
        global_features: torch.Tensor,
        candidate_mask: torch.Tensor,
    ) -> PolicyOutput:
        self_hidden = self.self_encoder(self_features)
        global_hidden = self.global_encoder(global_features)
        candidate_hidden = self.candidate_encoder(candidate_features)
        expanded_self = self_hidden.unsqueeze(1).expand(-1, self.candidate_count, -1)
        expanded_global = global_hidden.unsqueeze(1).expand(-1, self.candidate_count, -1)
        joint = torch.cat([expanded_self, expanded_global, candidate_hidden], dim=-1)
        target_logits = self.target_head(joint).squeeze(-1)
        target_logits = target_logits.masked_fill(~candidate_mask, torch.finfo(target_logits.dtype).min)
        pooled_candidates = candidate_hidden.mean(dim=1)
        value = self.value_head(torch.cat([self_hidden, global_hidden, pooled_candidates], dim=-1)).squeeze(-1)
        return PolicyOutput(target_logits=target_logits, value=value)

In [ ]:
# ppo.py
from __future__ import annotations

import torch
from torch.distributions import Categorical
from dataclasses import dataclass
# from .policy import PolicyOutput


@dataclass(slots=True)
class SampledAction:
  target_index: torch.Tensor
  log_prob: torch.Tensor
  entropy: torch.Tensor


@dataclass(slots=True)
class TransitionBatch:
  self_features: torch.Tensor
  candidate_features: torch.Tensor
  global_features: torch.Tensor
  candidate_mask: torch.Tensor
  target_index: torch.Tensor
  log_prob: torch.Tensor
  returns: torch.Tensor
  advantages: torch.Tensor


# Calc. Entropy and log probability
def action_log_prob_and_entropy(
    outputs: PolicyOutput,
    target_index: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:

  tgt_logits = safe_tgt_logits(outputs.target_logits)
  tgt_dist = Categorical(logits=tgt_logits)

  tgt_log_prob = tgt_dist.log_prob(target_index)
  tgt_entropy = tgt_dist.entropy()

  return tgt_log_prob, tgt_entropy


# Fix invalid logits
def safe_tgt_logits(target_logits: torch.Tensor) -> torch.Tensor:
  invalid_rows = ~torch.isfinite(target_logits).any(dim=-1)

  if not invalid_rows.any():
    return target_logits

  safe_logits = target_logits.clone()
  safe_logits[invalid_rows, 0] = 0.0
  return safe_logits


# choose an action from policy o/p and return SampledAction obj
def sample_actions(outputs: PolicyOutput, deterministic: bool) -> SampledAction:

  # getting target_idx
  tgt_logits = safe_tgt_logits(outputs.target_logits)
  tgt_dist = Categorical(logits=tgt_logits)

  tgt_idx = (
      target_logits.argmax(dim=-1)
      if deterministic
      else tgt_dist.sample()
  )

  # cal. log prob and entropy
  log_prob, entropy = action_log_prob_and_entropy(outputs, tgt_idx)

  return SampledAction(
      target_index=tgt_idx,
      log_prob=log_prob,
      entropy=entropy
  )


# Actual ppo training step
def ppo_update(
    policy: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    batch: TransitionBatch,
    *,
    clip_coef: float,
    ent_coef: float,
    vf_coef: float,
    max_grad_norm: float,
    epochs: int,
    minibatch_size: int,
    device: torch.device,
) -> dict[str, float]:

  # Empty batch check
  if batch.self_features.shape[0] == 0:
    return {
        "loss": 0.0,
        "policy_loss": 0.0,
        "value_loss": 0.0,
        "entropy": 0.0
    }

  # Move tensors to device
  self_feat = batch.self_features.to(device)
  candidate_feat = batch.candidate_features.to(device)
  global_feat = batch.global_features.to(device)
  candidate_mask = batch.candidate_mask.to(device).bool()
  old_log_prob = batch.log_prob.to(device)
  tgt_idx = batch.target_index.to(device)
  returns = batch.returns.to(device)
  adv = batch.advantages.to(device)

  # Adv normalization
  adv = (adv - adv.mean()) / (adv.std(unbiased=False) + 1e-8)

  # MiniBatchs
  size = self_feat.shape[0]
  minibatch_size = min(size, max(1, minibatch_size))

  metrics = {
      "loss": 0.0,
      "policy_loss": 0.0,
      "value_loss": 0.0,
      "entropy": 0.0
  }

  updates = 0

  for _ in range(epochs):

    # shuffle
    order = torch.randperm(size, device=device)

    for start in range(0, size, minibatch_size):
      idx = order[start:start + minibatch_size]

      # forward pass - gives logits, values
      outputs = policy(
          self_feat[idx],
          candidate_feat[idx],
          global_feat[idx],
          candidate_mask[idx]
      )

      # log prob. of chosen action
      new_log_prob, entropy = action_log_prob_and_entropy(outputs, tgt_idx[idx])

      # PPO ratio
      ratio = (new_log_prob - old_log_prob[idx]).exp()

      # Policy loss
      policy_loss = torch.maximum(
          -adv[idx] * ratio,
          -adv[idx] * torch.clamp(ratio, 1.0 - clip_coef, 1.0 + clip_coef)
      ).mean()

      # Value loss
      value_loss = 0.5 * (returns[idx] - outputs.value).pow(2).mean()

      entropy = entropy.mean()

      loss = policy_loss + vf_coef * value_loss - ent_coef * entropy

      # Backprop
      optimizer.zero_grad(set_to_none=True)
      loss.backward()

      torch.nn.utils.clip_grad_norm_(policy.parameters(), max_grad_norm)
      optimizer.step()

      # Metrics accumulation
      metrics["loss"] += float(loss.detach().cpu())
      metrics["policy_loss"] += float(policy_loss.detach().cpu())
      metrics["value_loss"] += float(value_loss.detach().cpu())
      metrics["entropy"] += float(entropy.detach().cpu())

      updates += 1

  return {key: value / max(updates, 1) for key, value in metrics.items()}

In [ ]:
# opponents.py
from __future__ import annotations

from typing import Any, Protocol

import torch


class OpponentPolicy(Protocol):
    def act(self, observation: Any) -> list[list[float | int]]:
        ...


class KaggleRandomOpponent:
    def __init__(self) -> None:
        from kaggle_environments.envs.orbit_wars.orbit_wars import random_agent

        self._agent = random_agent

    def act(self, observation: Any) -> list[list[float | int]]:
        payload = {
            "player": obs_get(observation, "player", 0),
            "planets": list(obs_get(observation, "planets", [])),
        }
        return list(self._agent(payload))


class SelfPlayOpponent:
    def __init__(self, cfg: TrainConfig, device: torch.device, deterministic: bool = True) -> None:
        # from .features import candidate_feature_dim, global_feature_dim, self_feature_dim

        self.cfg = cfg
        self.device = device
        self.deterministic = deterministic
        self.policy = PlanetPolicy(
            self_dim=self_feature_dim(),
            candidate_dim=candidate_feature_dim(),
            global_dim=global_feature_dim(),
            candidate_count=cfg.env.candidate_count,
            hidden_size=cfg.model.hidden_size,
        ).to(device)
        self.policy.eval()

    def sync_from(self, source_policy: PlanetPolicy) -> None:
        self.policy.load_state_dict(source_policy.state_dict())
        self.policy.eval()

    def act(self, observation: Any) -> list[list[float | int]]:
        batch = encode_turn(observation, self.cfg.env, env_index=0)
        if batch.self_features.shape[0] == 0:
            return []
        with torch.inference_mode():
            outputs = self.policy(
                torch.from_numpy(batch.self_features).to(self.device),
                torch.from_numpy(batch.candidate_features).to(self.device),
                torch.from_numpy(batch.global_features).to(self.device),
                torch.from_numpy(batch.candidate_mask).to(self.device).bool(),
            )
            sampled = sample_actions(outputs, deterministic=self.deterministic)
        target_indices = sampled.target_index.detach().cpu().numpy()
        moves: list[list[float | int]] = []
        for row_idx, context in enumerate(batch.contexts):
            target_idx = int(target_indices[row_idx])
            if target_idx == 0:
                continue
            if target_idx >= len(context.candidate_ids):
                continue
            if not context.candidate_mask[target_idx]:
                continue
            ships = int(context.ship_counts[target_idx])
            if ships <= 0:
                continue
            moves.append([context.source_id, float(context.target_angles[target_idx]), ships])
        return moves


def build_opponent(
    name: str,
    cfg: TrainConfig | None = None,
    device: torch.device | None = None,
) -> OpponentPolicy:
    if name == "random":
        return KaggleRandomOpponent()
    if name == "self":
        if cfg is None or device is None:
            raise ValueError("cfg and device are required for self opponent")
        return SelfPlayOpponent(cfg, device=device, deterministic=cfg.self_play_deterministic)
    raise ValueError(f"Unknown opponent: {name}")


def obs_get(observation: Any, key: str, default: Any) -> Any:
    if isinstance(observation, dict):
        return observation.get(key, default)
    return getattr(observation, key, default)

In [ ]:
# env.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any


@dataclass(slots=True)
class StepResult:
    batch: TurnBatch
    reward: float
    done: bool
    info: dict[str, Any]

class OrbitWarsEnv:
    def __init__(
        self,
        cfg: TrainConfig,
        opponent: OpponentPolicy,
        make_fn: Any | None = None,
        env_index: int = 0,
    ) -> None:
        self.cfg = cfg
        self.opponent = opponent
        self.make_fn = make_fn
        self.env_index = env_index

        self.env: Any | None = None
        self.last_obs: Any | None = None
        self.last_opp_obs: Any | None = None

        self.episode_index = 0
        self.learner_player = 0

    def reset(self, seed: int | None = None) -> TurnBatch:
        make_fn = self.make_fn or default_make_fn()

        config: dict[str, Any] = {}
        if seed is not None:
            config["seed"] = seed
            config["randomSeed"] = seed

        if self.cfg.alternate_player_sides:
            self.learner_player = (self.env_index + self.episode_index) % 2
        else:
            self.learner_player = 0

        self.env = make_fn("orbit_wars", configuration=config, debug=False)
        self.env.reset(num_agents=2)

        states = self.env.step([[], []])

        learner_state = states[self.learner_player]
        opp_state = states[1 - self.learner_player]

        self.last_obs = extract_observation(learner_state)
        self.last_opp_obs = extract_observation(opp_state)

        self.episode_index += 1

        return encode_turn(self.last_obs, self.cfg.env, env_index=self.env_index)

    def step(self, player_action: list[list[float | int]]) -> StepResult:
        if self.env is None:
            raise RuntimeError("Environment not initialized. Call reset().")

        opponent_action = self.opponent.act(self.last_opp_obs)

        if self.learner_player == 0:
            actions = [player_action, opponent_action]
        else:
            actions = [opponent_action, player_action]

        states = self.env.step(actions)

        player_state = states[self.learner_player]
        opp_state = states[1 - self.learner_player]

        self.last_obs = extract_observation(player_state)
        self.last_opp_obs = extract_observation(opp_state)

        done = extract_status(player_state) != "ACTIVE"
        reward = terminal_reward(player_state, opp_state) if done else 0.0

        batch = encode_turn(self.last_obs, self.cfg.env, env_index=self.env_index)

        info = {
            "learner_player": self.learner_player,
            "player_status": extract_status(player_state),
            "opponent_status": extract_status(opp_state),
            "reward": reward,
        }

        return StepResult(batch=batch, reward=reward, done=done, info=info)

def default_make_fn() -> Any:
    from kaggle_environments import make
    return make


def extract_observation(state: Any) -> Any:
    if isinstance(state, dict):
        return state.get("observation")
    return getattr(state, "observation", None)


def extract_status(state: Any) -> str:
    if isinstance(state, dict):
        return str(state.get("status", "UNKNOWN"))
    return str(getattr(state, "status", "UNKNOWN"))


def extract_reward(state: Any) -> float:
    if isinstance(state, dict):
        value = state.get("reward", 0.0)
    else:
        value = getattr(state, "reward", 0.0)

    return 0.0 if value is None else float(value)


def terminal_reward(player_state: Any, opp_state: Any) -> float:
    player_reward = extract_reward(player_state)
    opponent_reward = extract_reward(opp_state)

    if player_reward > 0.0 and opponent_reward > 0.0:
        return 0.0

    return player_reward

In [ ]:
from __future__ import annotations

import argparse
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch

CONFIG_PATH = "/content/default_cfg.yaml"

@dataclass(slots=True)
class StepGroup:
  indices: list[int]
  reward: float
  done: bool



# Device agnostic code
def resolve_device(name: str) -> torch.device:
  if name == "auto":
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")
  return torch.device(name)


# Random seed - Reproducibility
def seed_everything(seed: int) -> None:
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


def bootstrap_values(policy: PlanetPolicy, batches: list[TurnBatch], device: torch.device) -> list[float]:
  merged = merge_batches(batches)
  if merged.self_features.shape[0] == 0:
    return [0.0 for _ in batches]

  offsets = np.cumsum([0] + [batch.self_features.shape[0] for batch in batches[:-1]])

  with torch.inference_mode():
    outputs = policy(
        torch.from_numpy(merged.self_features).to(device),
        torch.from_numpy(merged.candidate_features).to(device),
        torch.from_numpy(merged.global_features).to(device),
        torch.from_numpy(merged.candidate_mask).to(device).bool(),
    )

  values = outputs.value.detach().cpu().numpy()

  per_env = []
  for env_idx, batch in enumerate(batches):
    start = int(offsets[env_idx])
    count = batch.self_features.shape[0]
    per_env.append(0.0 if count == 0 else float(values[start:start + count].mean()))

  return per_env


def merge_batches(batches: list[TurnBatch]) -> TurnBatch:
  if not batches:
    raise ValueError("batches must not be empty")

  has_rows = any(batch.self_features.shape[0] > 0 for batch in batches)

  self_rows = (
      np.concatenate([batch.self_features for batch in batches], axis=0)
      if has_rows
      else np.zeros((0, self_feature_dim()), dtype=np.float32)
  )

  candidate_rows = (
      np.concatenate([batch.candidate_features for batch in batches], axis=0)
      if has_rows
      else np.zeros(
          (0, batches[0].candidate_features.shape[1], candidate_feature_dim()),
          dtype=np.float32,
      )
  )

  global_rows = (
      np.concatenate([batch.global_features for batch in batches], axis=0)
      if has_rows
      else np.zeros((0, global_feature_dim()), dtype=np.float32)
  )

  candidate_masks = (
      np.concatenate([batch.candidate_mask for batch in batches], axis=0)
      if has_rows
      else np.zeros((0, batches[0].candidate_mask.shape[1]), dtype=bool)
  )

  return TurnBatch(
      self_features=self_rows,
      candidate_features=candidate_rows,
      global_features=global_rows,
      candidate_mask=candidate_masks,
      contexts=[context for batch in batches for context in batch.contexts],
      state=batches[0].state,
  )


def save_checkpoint(
    save_dir: Path,
    run_name: str,
    update: int,
    policy: PlanetPolicy,
    optimizer: torch.optim.Optimizer,
    cfg: TrainConfig,
) -> None:
  run_dir = save_dir / run_name
  run_dir.mkdir(parents=True, exist_ok=True)

  torch.save(
      {
          "update": update,
          "policy": policy.state_dict(),
          "optimizer": optimizer.state_dict(),
          "config": cfg,
      },
      run_dir / "ckpt_last.pt",
  )

  torch.save(
      {
          "update": update,
          "policy": policy.state_dict(),
          "optimizer": optimizer.state_dict(),
          "config": cfg,
      },
      run_dir / f"ckpt_{update:06d}.pt",
  )


def find_planet(planets: list[PlanetState], planet_id: int) -> PlanetState | None:
  for planet in planets:
    if planet.id == planet_id:
      return planet
  return None


def collect_rollouts(
    envs: list[OrbitWarsEnv],
    batches: list[TurnBatch],
    policy: PlanetPolicy,
    cfg: TrainConfig,
    device: torch.device,
    nxt_seed: int,
) -> tuple[TransitionBatch, list[TurnBatch], int, dict[str, float]]:
  empty_candidate = (cfg.env.candidate_count, candidate_feature_dim())
  self_rows: list[np.ndarray] = []
  candidate_rows: list[np.ndarray] = []
  global_rows: list[np.ndarray] = []
  candidate_masks: list[np.ndarray] = []
  target_indices: list[int] = []
  log_probs: list[float] = []
  values: list[float] = []
  groups_per_env: list[list[StepGroup]] = [[] for _ in envs]
  episode_rewards: list[float] = []
  running_episode_rewards = [0.0 for _ in envs]

  # Run for N steps
  for _ in range(cfg.ppo.rollout_steps):
    # env index mapping
    offsets = np.cumsum([0] + [batch.self_features.shape[0] for batch in batches[:-1]])

    merged = merge_batches(batches)

    row_values = np.zeros((merged.self_features.shape[0],), dtype=np.float32)

    # if data is there
    if merged.self_features.shape[0] > 0:
      with torch.inference_mode():
        outputs = policy(
            torch.from_numpy(merged.self_features).to(device),
            torch.from_numpy(merged.candidate_features).to(device),
            torch.from_numpy(merged.global_features).to(device),
            torch.from_numpy(merged.candidate_mask).to(device).bool(),
        )

        # sample actions from policy and extract o/p to numpy
        sampled = sample_actions(outputs, deterministic=False)
        row_values = outputs.value.detach().cpu().numpy()
        sampled_target_index = sampled.target_index.detach().cpu().numpy()
        sampled_log_prob = sampled.log_prob.detach().cpu().numpy()
    else:
      # No state - empty arrays
      sampled_target_index = np.zeros((0,), dtype=np.int64)
      sampled_log_prob = np.zeros((0,), dtype=np.float32)

    next_batches: list[TurnBatch] = []
    for env_idx, env in enumerate(envs):
      batch = batches[env_idx]
      start = int(offsets[env_idx])
      moves = []
      group_indices: list[int] = []

      for local_idx, context in enumerate(batch.contexts):
        # local idx -> global idx
        global_idx = start + local_idx

        # store training data
        self_rows.append(batch.self_features[local_idx])
        candidate_rows.append(batch.candidate_features[local_idx])
        global_rows.append(batch.global_features[local_idx])
        candidate_masks.append(batch.candidate_mask[local_idx])
        values.append(float(row_values[global_idx]))

        # action from policy o/p
        tgt_idx = (
            sampled_target_index[global_idx]
            if batch.self_features.shape[0] > 0
            else 0
        )

        # action validity
        is_valid_send = (
            tgt_idx > 0
            and tgt_idx < len(context.candidate_ids)
            and context.candidate_mask[tgt_idx]
            and int(context.ship_counts[tgt_idx]) > 0
        )

        target_indices.append(tgt_idx)
        log_probs.append(
            float(sampled_log_prob[global_idx])
            if batch.self_features.shape[0] > 0
            else 0.0
        )

        group_indices.append(len(values) - 1)

        if not is_valid_send:
          continue

        ships = int(context.ship_counts[tgt_idx])
        src_planet = find_planet(batch.state.planets, context.source_id)

        if src_planet is None or src_planet.ships < ships:
          continue

        moves.append([context.source_id, float(context.target_angles[tgt_idx]), ships])

      result = env.step(moves)

      # rewards
      running_episode_rewards[env_idx] += float(result.reward)

      # group info
      groups_per_env[env_idx].append(
          StepGroup(indices=group_indices, reward=float(result.reward), done=result.done)
      )

      if result.done:
        episode_rewards.append(running_episode_rewards[env_idx])
        running_episode_rewards[env_idx] = 0.0
        nxt_seed += 1
        next_batch = env.reset(seed=nxt_seed)
      else:
        next_batch = result.batch

      next_batches.append(next_batch)

    batches = next_batches

  returns: list[float] = [0.0] * len(values)
  advantages: list[float] = [0.0] * len(values)

  # bootstrap last state
  next_state_values = bootstrap_values(policy, batches, device)

  # Backpropogate rewards in time
  for env_idx, groups in enumerate(groups_per_env):
    future_return = next_state_values[env_idx]

    for group in reversed(groups):
      # update rule
      future_return = group.reward + cfg.ppo.gamma * future_return * (1.0 - float(group.done))

      for idx in group.indices:
        returns[idx] = future_return
        advantages[idx] = future_return - values[idx]

  # for ppo training
  batch = TransitionBatch(
      self_features=torch.from_numpy(
          np.asarray(self_rows, dtype=np.float32).reshape(-1, self_feature_dim())
      ),
      candidate_features=torch.from_numpy(
          np.asarray(candidate_rows, dtype=np.float32).reshape(
              -1, empty_candidate[0], empty_candidate[1]
          )
      ),
      global_features=torch.from_numpy(
          np.asarray(global_rows, dtype=np.float32).reshape(-1, global_feature_dim())
      ),
      candidate_mask=torch.from_numpy(
          np.asarray(candidate_masks, dtype=bool).reshape(-1, cfg.env.candidate_count)
      ),
      target_index=torch.tensor(target_indices, dtype=torch.long),
      log_prob=torch.tensor(log_probs, dtype=torch.float32),
      returns=torch.tensor(returns, dtype=torch.float32),
      advantages=torch.tensor(advantages, dtype=torch.float32),
  )

  # Logging
  stats = {
      "episode_reward_mean": float(np.mean(episode_rewards)) if episode_rewards else 0.0,
      "episodes_finished": float(len(episode_rewards)),
      "samples": float(len(values)),
  }

  return batch, batches, nxt_seed, stats


def main() -> None:
  # load config
  cfg = load_train_config(CONFIG_PATH)

  # reproducibility + device selection
  seed_everything(cfg.seed)
  device = resolve_device(cfg.device)

  # opponent + vectorized env
  opponent = build_opponent(cfg.opponent, cfg=cfg, device=device)
  envs = [OrbitWarsEnv(cfg, opponent, env_index=idx) for idx in range(cfg.ppo.num_envs)]

  nxt_seed = cfg.seed
  batches = []
  for env in envs:
    batches.append(env.reset(seed=nxt_seed))
    nxt_seed += 1

  # policy network
  policy = PlanetPolicy(
      self_dim=self_feature_dim(),
      candidate_dim=candidate_feature_dim(),
      global_dim=global_feature_dim(),
      candidate_count=cfg.env.candidate_count,
      hidden_size=cfg.model.hidden_size,
  ).to(device)

  # self play setup
  if isinstance(opponent, SelfPlayOpponent):
    opponent.sync_from(policy)

  # optimizer
  optimizer = torch.optim.Adam(policy.parameters(), lr=cfg.ppo.lr)

  save_dir = Path(cfg.save_dir)

  # PPO loop - collect rollout, train ppo, save, repeat
  for update in range(1, cfg.ppo.total_updates + 1):
    batch, batches, nxt_seed, stats = collect_rollouts(
        envs, batches, policy, cfg, device, nxt_seed
    )

    metrics = ppo_update(
        policy,
        optimizer,
        batch,
        clip_coef=cfg.ppo.clip_coef,
        ent_coef=cfg.ppo.ent_coef,
        vf_coef=cfg.ppo.vf_coef,
        max_grad_norm=cfg.ppo.max_grad_norm,
        epochs=cfg.ppo.epochs,
        minibatch_size=cfg.ppo.minibatch_size,
        device=device,
    )

    # self play
    if isinstance(opponent, SelfPlayOpponent) and update % cfg.self_play_update_interval == 0:
      opponent.sync_from(policy)

    # logging
    if update % cfg.log_every == 0:
      print(
          f"update={update} episode_reward_mean={stats['episode_reward_mean']:.4f} "
          f"episodes={int(stats['episodes_finished'])} samples={int(stats['samples'])} "
          f"loss={metrics['loss']:.4f}"
      )

    # save model
    if update % cfg.checkpoint_every == 0 or update == cfg.ppo.total_updates:
      save_checkpoint(save_dir, cfg.run_name, update, policy, optimizer, cfg)





main()

In [ ]:
# Agent Bot
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import (
    Planet,
Fleet)

# Board size
BOARD = 100.0

# Sun position (center of map)
CENTER_X = 50.0
CENTER_Y = 50.0

# Sun radius
SUN_R = 10.0

# Maximum fleet speed possible
MAX_SPEED = 6.0

# Extra safety distance around sun
SUN_SAFETY = 1.5

# Outer planets rotate less
ROTATION_LIMIT = 50.0

# Total game duration
TOTAL_STEPS = 500

def fleet_speed(
    num_ships: int,
    max_speed: float = MAX_SPEED
) -> float:


    # Prevent invalid or zero fleets
    if num_ships <= 1:
        return 1.0

    # Convert ship count into value between 0 and 1
    ratio = math.log(num_ships) / math.log(1000)

    # Curved speed scaling
    speed = 1.0 + (
        max_speed - 1.0
    ) * (ratio ** 1.5)

    return speed

def dist(
    x1: float,
    y1: float,
    x2: float,
    y2: float
) -> float:

    return math.hypot(
        x2 - x1,
        y2 - y1
    )


def travel_time(
    x1: float,
    y1: float,
    x2: float,
    y2: float,
    ships: int
) -> float:

    # Compute distance
    d = dist(
        x1,
        y1,
        x2,
        y2
    )

    # Prevent divide-by-zero
    if ships <= 0:
        return 999.0

    # Fleet speed depends on fleet size
    speed = fleet_speed(ships)

    # Final travel time
    return d / speed

def future_production(
    planet: Planet,
    turns: float
) -> int:

    return int(
        math.ceil(
            planet.production * turns
        )
    )

def ships_needed_to_capture(
    target: Planet,
    travel_turns: float,
    safety_margin: float = 1.05
) -> int:


    # Future defenders at arrival time
    future_garrison = (
        target.ships
        + future_production(
            target,
            travel_turns
        )
    )

    # Add safety buffer
    needed = int(
        math.ceil(
            future_garrison * safety_margin
        )
    ) + 1

    return max(1, needed)

def net_def_needed(
    planet: Planet,
    incoming_fleets: list,
    player_id: int,
    safety_margin: float = 1.15
) -> int:


    # Current defending ships
    garrison = planet.ships

    # Largest future deficit
    max_deficit = 0

    # Sort fleets by arrival time
    events = sorted(
        incoming_fleets,
        key=lambda x: x[0]
    )

    # Process all future arrivals
    for eta, owner, ships in events:

        # Planet produces ships until fleet arrives
        garrison += (
            eta * planet.production
        )

        # Enemy fleet attacks
        if owner != player_id:

            # Reduce defenders
            garrison -= ships

            # Planet would fall
            if garrison < 0:

                # Track largest deficit
                max_deficit = max(
                    max_deficit,
                    -garrison
                )

    # Add extra defensive safety
    reserve = int(
        math.ceil(
            max_deficit * safety_margin
        )
    )

    return max(0, reserve)

# Sun hits either sun or planet

def segment_hits_circle(x1, y1, x2, y2, cx, cy, radius, safety=0.0):
    r = radius + safety
    dx, dy = x2 - x1, y2 - y1
    fx, fy = x1 - cx, y1 - cy
    a = dx * dx + dy * dy


    if max(x1, x2) < cx - r:
        return False
    if min(x1, x2) > cx + r:
        return False
    if max(y1, y2) < cy - r:
        return False
    if min(y1, y2) > cy + r:
        return False

    if a < 1e-9:
        return math.hypot(x1 - cx, y1 - cy) < r

    b = 2 * (fx * dx + fy * dy)
    c = fx * fx + fy * fy - r * r

    disc = b * b - 4 * a * c
    if disc < 0:
        return False

    disc = math.sqrt(disc)
    t1 = (-b - disc) / (2 * a)
    t2 = (-b + disc) / (2 * a)

    return (0 <= t1 <= 1) or (0 <= t2 <= 1)


# Fleet which are coming for our planets

def incoming_to_planet(planet, fleets):
    arrivals = []
    fvx_cache = {}
    for f in fleets:
        fvx, fvy = math.cos(f.angle), math.sin(f.angle)
        dx, dy = planet.x - f.x, planet.y - f.y
        proj = dx * fvx + dy * fvy
        if proj <= 0:
            continue
        perp = abs(dx * fvy - dy * fvx)
        if perp > planet.radius + 1.5:
            continue
        t = proj / fleet_speed(f.ships)
        if t > 80:
            continue
        arrivals.append((int(math.ceil(t)), f.owner, int(f.ships)))
    return arrivals


# Predict planets position
def predict_planet_position(planet, initial_by_id, angular_velocity, turns):
    init = initial_by_id.get(planet.id)
    if init is None:
        return planet.x, planet.y
    orbital_r = dist(init.x, init.y, CENTER_X, CENTER_Y)
    if orbital_r + init.radius >= ROTATION_LIMIT:
        return planet.x, planet.y
    cur_ang = math.atan2(planet.y - CENTER_Y, planet.x - CENTER_X)
    new_ang = cur_ang + angular_velocity * turns
    return (CENTER_X + orbital_r * math.cos(new_ang),
            CENTER_Y + orbital_r * math.sin(new_ang))




# Comet helper functions
def comet_remaining_life(planet_id, comets):
  for group in comets:
    # finding the group which got planet_id
    pids = group.get("planet_ids", [])
    if planet_id not in pids:
      continue

    # get the idx of our planet_id
    idx = pids.index(planet_id)

    # All the paths -- we want paths of our planetid
    paths = group.get("paths", [])

    # Current position
    path_idx = group.get("path_index", 0)

    if idx < len(paths):
      return max(0, len(paths[idx]) - path_idx)

  return 0


def predict_comet_position(planet_id, comets, turns):
  for group in comets:
    pids = group.get("planet_ids", [])
    if planet_id not in pids:
      continue

    idx = pids.index(planet_id)
    paths = group.get("paths", [])
    path_idx = group.get("path_index", 0)

    # Paths of our planet id
    path = paths[idx]

    # Bound check
    if idx >= len(paths):
      return None

    future_idx = path_idx + int(turns)
    if 0 <= future_idx < len(path):
      return path[future_idx][0], path[future_idx][1]
    return None
  return None


def comet_gain(planet_id, comets, ships_needed, travel_time, remaining_life):

  if travel_time >= remaining_life:
    return -10**9

  usable_turns = remaining_life - travel_time
  future_ships = usable_turns * 1
  gain = future_ships - ships_needed

  return gain


def evacuate_expiring_comets(my_planets, comet_ids, comets, initial_by_id, angular_velocity):
  moves = []
  evacuated_srcs = set()

  safe_planets = [p for p in my_planets if p.id not in comet_ids]
  if not safe_planets:
    return moves, evacuated_srcs

  for comet in my_planets:
    if comet.id not in comet_ids:
      continue
    if comet.ships <= 0:
      continue

    remaining_life = comet_remaining_life(comet.id, comets)
    if remaining_life > 3:
      continue

    dest = min(safe_planets, key=lambda p: dist(comet.x, comet.y, p.x, p.y))

    travel_turns = travel_time(comet.x, comet.y, dest.x, dest.y, comet.ships)
    if travel_turns >= remaining_life:
      continue

    fut_x, fut_y = predict_planet_position(dest, initial_by_id, angular_velocity, travel_turns)

    angle = math.atan2(fut_y - comet.y, fut_x - comet.x)

    if segment_hits_circle(comet.x, comet.y, fut_x, fut_y, 50.0, 50.0, 10.0, safety=0.5):
      continue

    moves.append([comet.id, float(angle), int(comet.ships)])
    evacuated_srcs.add(comet.id)

  return moves, evacuated_srcs


# Agent

import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

TOTAL_STEPS = 500

def agent(obs):
    moves = []

    # Extracting info
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    step = obs.get("step", 0) if isinstance(obs, dict) else obs.step

    planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    fleets = obs.get("fleets", []) if isinstance(obs, dict) else obs.fleets
    angular_velocity = obs.get("angular_velocity", 0) if isinstance(obs, dict) else obs.angular_velocity

    initial_planets = obs.get("initial_planets", []) if isinstance(obs, dict) else obs.initial_planets
    comets = obs.get("comets", []) if isinstance(obs, dict) else obs.comets
    comet_ids = obs.get("comet_planet_ids", []) if isinstance(obs, dict) else obs.comet_planet_ids

    # Converting into obj
    planets = [Planet(*p) for p in planets]
    fleets = [Fleet(*f) for f in fleets]

    planet_by_id = {p.id: p for p in planets}

    # Initial positions
    initial_by_id = {
        Planet(*p).id: Planet(*p)
        for p in initial_planets
    }

    # Splitting planets - mine vs target
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not my_planets:
        return []

    # Evacuate comets
    evac_moves, evac_srcs = evacuate_expiring_comets(
        my_planets,
        comet_ids,
        comets,
        initial_by_id,
        angular_velocity
    )
    moves.extend(evac_moves)

    # Defining game phases
    rem_steps = max(1, TOTAL_STEPS - step)

    early_phase = step < 60
    late_phase = rem_steps < 60
    mid_phase = not early_phase and not late_phase

    # Ships available for attack
    available = {}

    for mine in my_planets:
        total_ships = mine.ships
        incoming_fleets = incoming_to_planet(mine, fleets)
        defence = net_def_needed(mine, incoming_fleets, player)
        available[mine.id] = max(0, total_ships - defence)

    # Candidates
    candidates = []
    taken_targets = set()

    for mine in my_planets:
        if available[mine.id] <= 0:
            continue

        for target in targets:

            comet = target.id in comet_ids
            neutral = target.owner == -1
            enemy = target.owner != player and target.owner != -1

            distance = dist(mine.x, mine.y, target.x, target.y)

            travel_turns = travel_time(
                mine.x, mine.y,
                target.x, target.y,
                available[mine.id]
            )

            ships_needed = ships_needed_to_capture(target, travel_turns)
            profit_turns = rem_steps - travel_turns

            if profit_turns <= 0:
                continue
            if ships_needed > available[mine.id]:
                continue

            # Normal planet
            if not comet:
                future_prod = future_production(target, profit_turns)

                attack_score = future_prod * 100.0
                attack_score /= (ships_needed + 1.0)
                attack_score /= (travel_turns + 1.0)
                attack_score /= (1.0 + distance * 0.05)
                attack_score *= (profit_turns / TOTAL_STEPS)

            # Comets
            else:
                remaining_life = comet_remaining_life(target.id, comets)
                if travel_turns >= remaining_life:
                    continue

                usable_turns = remaining_life - travel_turns
                if usable_turns <= ships_needed:
                    continue

                gain = comet_gain(
                    target.id,
                    comets,
                    ships_needed,
                    travel_turns,
                    remaining_life
                )

                attack_score = gain * 100.0
                attack_score /= (1.0 + distance * 0.05)

            # ignore enemy comets
            if enemy and comet:
                continue

            # Phase adjustments
            if early_phase:
                if neutral and not comet:
                    attack_score *= 2.0
                elif neutral and comet:
                    attack_score *= 1.5
                elif enemy and not comet:
                    attack_score *= 0.6

            elif mid_phase:
                if enemy and not comet:
                    attack_score *= 1.2

            elif late_phase:
                if comet:
                    continue
                if enemy and not comet:
                    attack_score *= 2.5
                elif neutral and not comet:
                    attack_score *= 0.5

            candidates.append(
                (attack_score, mine.id, target.id, ships_needed, travel_turns)
            )

    # sort
    candidates.sort(reverse=True)

    # Launch attack
    for attack_score, source_id, target_id, ships_needed, travel_turns in candidates:

        if target_id in taken_targets:
            continue
        if available[source_id] < ships_needed:
            continue

        src = planet_by_id[source_id]
        target = planet_by_id[target_id]

        if target.id in comet_ids:
            future_x, future_y = predict_comet_position(
                target.id,
                comets,
                travel_turns
            )
        else:
            future_x, future_y = predict_planet_position(
                target,
                initial_by_id,
                angular_velocity,
                travel_turns
            )

        # Angle
        angle = math.atan2(future_y - src.y, future_x - src.x)

        # Sun check
        if segment_hits_circle(
            src.x, src.y,
            future_x, future_y,
            50.0, 50.0, 10.0,
            safety=0.5
        ):
            continue

        moves.append([
            source_id,
            float(angle),
            int(ships_needed)
        ])

        available[source_id] -= ships_needed
        taken_targets.add(target_id)

    return moves


In [ ]:
# Competing RL vs Heuristic Agent
from __future__ import annotations

import argparse
import math
import random
from collections import namedtuple
from typing import Any

import numpy as np
import torch

CONFIG_PATH = "/content/default_cfg.yaml"

Planet = namedtuple("Planet", ["id", "owner", "x", "y", "radius", "ships", "production"])

def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument("--config", type=str, default=CONFIG_PATH)
    parser.add_argument("--checkpoint", type=str, default=None)
    parser.add_argument("--games", type=int, default=10)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", type=str, default="auto")
    parser.add_argument("--deterministic", action="store_true")

    args, _ = parser.parse_known_args()
    return args


def resolve_device(name: str):
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(name)


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Policy Builder
def build_policy(cfg, device):
    return PlanetPolicy(
        self_dim=self_feature_dim(),
        candidate_dim=candidate_feature_dim(),
        global_dim=global_feature_dim(),
        candidate_count=cfg.env.candidate_count,
        hidden_size=cfg.model.hidden_size,
    ).to(device)

def load_checkpoint(policy, path, device):
    if path is None:
        return
    ckpt = torch.load(path, map_location=device)
    state = ckpt.get("policy", ckpt)
    policy.load_state_dict(state)


# Move BUilder
def build_moves(batch, policy, device, deterministic):
    if batch.self_features.shape[0] == 0:
        return []

    with torch.inference_mode():
        outputs = policy(
            torch.from_numpy(batch.self_features).to(device),
            torch.from_numpy(batch.candidate_features).to(device),
            torch.from_numpy(batch.global_features).to(device),
            torch.from_numpy(batch.candidate_mask).to(device).bool(),
        )

        sampled = sample_actions(outputs, deterministic=deterministic)

    targets = sampled.target_index.detach().cpu().numpy()

    moves = []
    for i, ctx in enumerate(batch.contexts):
        t = int(targets[i])

        if t <= 0 or t >= len(ctx.candidate_ids):
            continue
        if not ctx.candidate_mask[t]:
            continue

        ships = int(ctx.ship_counts[t])
        if ships <= 0:
            continue

        moves.append([ctx.source_id, float(ctx.target_angles[t]), ships])

    return moves


# Sniper
def nearest_sniper(obs):
    moves = []

    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets

    planets = [Planet(*p) for p in planets]

    mine = [p for p in planets if p.owner == player]
    enemy = [p for p in planets if p.owner != player]

    if not enemy:
        return moves

    for m in mine:
        target = min(enemy, key=lambda e: math.hypot(m.x - e.x, m.y - e.y))

        needed = max(target.ships + 1, 20)
        if m.ships < needed:
            continue

        angle = math.atan2(target.y - m.y, target.x - m.x)
        moves.append([m.id, angle, needed])

    return moves



def extract_obs(state):
    return state.get("observation") if isinstance(state, dict) else state.observation


def extract_status(state):
    return state.get("status", "UNKNOWN") if isinstance(state, dict) else state.status


def extract_reward(state):
    r = state.get("reward", 0.0) if isinstance(state, dict) else state.reward
    return 0.0 if r is None else float(r)


# Game loop
def play_game(cfg, policy, device, seed, deterministic):
    from kaggle_environments import make

    env = make("orbit_wars", configuration={"seed": seed}, debug=False)
    env.reset(num_agents=2)

    states = env.step([[], []])

    obs_p = extract_obs(states[0])
    obs_e = extract_obs(states[1])

    done = False
    steps = 0

    while not done:
        batch = encode_turn(obs_p, cfg.env, env_index=0)

        a1 = build_moves(batch, policy, device, deterministic)
        a2 = agent(obs_e)

        states = env.step([a1, a2])

        obs_p = extract_obs(states[0])
        obs_e = extract_obs(states[1])

        done = extract_status(states[0]) != "ACTIVE"
        steps += 1

    reward = extract_reward(states[0])
    return reward, steps


def main():
    args = parse_args()

    cfg = load_train_config(args.config)

    device = resolve_device(args.device)
    seed_everything(args.seed)

    policy = build_policy(cfg, device)
    load_checkpoint(policy, args.checkpoint, device)
    policy.eval()

    wins = losses = draws = 0

    for i in range(args.games):
        reward, steps = play_game(
            cfg,
            policy,
            device,
            seed=args.seed + i,
            deterministic=args.deterministic,
        )

        if reward > 0:
            wins += 1
            result = "win"
        elif reward < 0:
            losses += 1
            result = "loss"
        else:
            draws += 1
            result = "draw"

        print(f"game={i+1} result={result} reward={reward:.2f} steps={steps}")

    print("\nFINAL SUMMARY")
    print("wins:", wins, "losses:", losses, "draws:", draws)
    print("win_rate:", wins / max(args.games, 1))


main()